## Generate Synthetic EHR Data

### Subtask:
Generate a synthetic dataset of 5000 patients, including a subset with sepsis patterns, comprising structured vitals and unstructured notes, formatted as an event log.


In [ ]:
!pip install faker

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from faker import Faker

# Initialize Faker
fake = Faker()

# Parameters
num_patients = 10000
sepsis_prob = 0.25

# Data storage
event_log = []

for patient_id in range(1, num_patients + 1):
    is_septic = random.random() < sepsis_prob

    # Simulate a stay duration between 24 and 48 hours
    stay_duration_hours = random.randint(24, 48)
    start_time = fake.date_time_this_year()

    # Base vitals
    base_hr = random.randint(60, 90)
    base_sbp = random.randint(110, 130)
    base_temp = 37.0

    for hour in range(stay_duration_hours):
        current_time = start_time + timedelta(hours=hour)

        # --- Vitals Generation ---
        hr = base_hr + np.random.normal(0, 5)
        sbp = base_sbp + np.random.normal(0, 5)
        temp = base_temp + np.random.normal(0, 0.2)

        if is_septic and hour > stay_duration_hours * 0.3: # Deteriorate after 30% of stay
            progression = (hour - (stay_duration_hours * 0.3)) / (stay_duration_hours * 0.7)
            hr += 30 * progression  # HR increases
            sbp -= 20 * progression # SBP decreases
            temp += 1.5 * progression # Temp increases

        # Add Vitals to Log
        event_log.append({
            'patient_id': patient_id,
            'timestamp': current_time,
            'event_type': 'Vitals',
            'variable': 'Heart Rate',
            'value': round(hr)
        })
        event_log.append({
            'patient_id': patient_id,
            'timestamp': current_time,
            'event_type': 'Vitals',
            'variable': 'Systolic BP',
            'value': round(sbp)
        })
        event_log.append({
            'patient_id': patient_id,
            'timestamp': current_time,
            'event_type': 'Vitals',
            'variable': 'Temperature',
            'value': round(temp, 1)
        })

        # --- Clinical Notes Generation (every ~4-6 hours) ---
        if hour % random.randint(4, 6) == 0:
            if is_septic and hour > stay_duration_hours * 0.3:
                notes = [
                    "Patient appears lethargic.",
                    "Fever spike noted, checking sepsis protocol.",
                    "Hypotension observing, fluids started.",
                    "Signs of deterioration, calling resident."
                ]
                note_text = random.choice(notes)
            else:
                notes = [
                    "Patient stable, resting comfortably.",
                    "Vitals within normal limits.",
                    "Patient sleeping.",
                    "No acute distress."
                ]
                note_text = random.choice(notes)

            event_log.append({
                'patient_id': patient_id,
                'timestamp': current_time,
                'event_type': 'Note',
                'variable': 'Clinical Note',
                'value': note_text
            })

# Convert to DataFrame
event_log_df = pd.DataFrame(event_log)

# Sort chronologically
event_log_df = event_log_df.sort_values(by=['patient_id', 'timestamp'])

# Display results
print(f"Total unique patients: {event_log_df['patient_id'].nunique()}")
event_log_df.head()

In [ ]:
# Verify patient count
unique_patients = event_log_df['patient_id'].nunique()
print(f"Verified unique patients: {unique_patients}")

# Display DataFrame info
print("\nDataFrame Info:")
event_log_df.info()

# # Define file path
# csv_path = os.path.join(project_dir, 'sepsis_event_log.csv')

# # Save to CSV
# event_log_df.to_csv(csv_path, index=False)
# print(f"\nEvent log saved to: {csv_path}")

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# 1. Identify Septic Patients
# Since the original 'is_septic' status wasn't saved in the dataframe, we'll randomly select 25% of patients to be our septic cohort.
# This ensures we have a specific list to apply the new logic to.
all_patient_ids = event_log_df['patient_id'].unique()
num_septic = int(len(all_patient_ids) * 0.25)
septic_patient_ids = set(np.random.choice(all_patient_ids, num_septic, replace=False))

print(f"Selected {len(septic_patient_ids)} patients for sepsis simulation (augmentation).")

new_events = []

# Group existing data by patient to get start/end times efficiently
patient_groups = event_log_df.groupby('patient_id')

for patient_id, group in patient_groups:
    is_septic = patient_id in septic_patient_ids

    # Get admission start time and duration
    start_time = group['timestamp'].min()
    end_time = group['timestamp'].max()
    duration = (end_time - start_time).total_seconds() / 3600 # hours

    # --- 2. Generate Lab Values ---
    # Labs are typically taken less frequently than vitals, e.g., every 12-24 hours
    current_lab_time = start_time
    while current_lab_time <= end_time:
        # Determine elapsed hours for progression logic
        elapsed_hours = (current_lab_time - start_time).total_seconds() / 3600

        # Baselines
        lactate = random.uniform(0.5, 1.5)
        creatinine = random.uniform(0.6, 1.2)
        bilirubin = random.uniform(0.1, 1.2)
        platelets = random.uniform(150, 400)

        # --- 3. Simulate Progression for Septic Patients ---
        if is_septic and elapsed_hours > (duration * 0.3):
             progression = (elapsed_hours - (duration * 0.3)) / (duration * 0.7)
             progression = min(max(progression, 0), 1) # Clamp between 0 and 1

             # Worsening values based on progression
             lactate += 2.0 * progression       # Increases > 2
             creatinine += 1.5 * progression    # Increases > 2
             bilirubin += 1.5 * progression     # Increases > 1.2
             platelets -= 100 * progression     # Decreases < 150

        # Add Lab Events
        labs = {
            'Lactate': round(lactate, 1),
            'Creatinine': round(creatinine, 2),
            'Bilirubin': round(bilirubin, 2),
            'Platelets': round(platelets)
        }

        for lab_name, lab_value in labs.items():
            new_events.append({
                'patient_id': patient_id,
                'timestamp': current_lab_time,
                'event_type': 'Lab',
                'variable': lab_name,
                'value': lab_value
            })

        # Next lab draw in 12-24 hours
        current_lab_time += timedelta(hours=random.randint(12, 24))

    # --- 4. Inject Trigger Phrases for Septic Patients ---
    if is_septic:
        # Add a specific note event at a random time after 30% of stay
        trigger_time = start_time + timedelta(hours=duration * 0.35 + random.uniform(0, duration * 0.2))
        if trigger_time < end_time:
             triggers = [
                 "Blood cultures drawn due to fever.",
                 "Broad spectrum antibiotics ordered.",
                 "Suspected infection, starting sepsis protocol.",
                 "Lactate elevated, fluids bolused."
             ]
             new_events.append({
                'patient_id': patient_id,
                'timestamp': trigger_time,
                'event_type': 'Note',
                'variable': 'Clinical Note',
                'value': random.choice(triggers)
             })

# --- 5. Append and Sort ---
new_events_df = pd.DataFrame(new_events)
augmented_event_log_df = pd.concat([event_log_df, new_events_df], ignore_index=True)

# Sort chronologically
augmented_event_log_df = augmented_event_log_df.sort_values(by=['patient_id', 'timestamp']).reset_index(drop=True)

# Update the main variable
event_log_df = augmented_event_log_df

print("Data augmentation complete.")
print(f"Added {len(new_events_df)} new lab and note events.")
event_log_df.head()

In [ ]:
import pandas as pd
import numpy as np

# 1. Prepare Data for Pivoting
# Separate numeric and text components to handle types correctly
numeric_mask = event_log_df['event_type'].isin(['Vitals', 'Lab'])
numeric_df = event_log_df[numeric_mask].copy()
numeric_df['value'] = pd.to_numeric(numeric_df['value'], errors='coerce')

notes_df = event_log_df[~numeric_mask].copy()

# Pivot tables
# Numeric: average values if multiple occur at same timestamp
df_num = numeric_df.pivot_table(index=['patient_id', 'timestamp'], columns='variable', values='value', aggfunc='mean')

# Notes: for initial pivot, just take the value. We handle aggregation in resampling.
df_notes = notes_df.pivot_table(index=['patient_id', 'timestamp'], columns='variable', values='value', aggfunc='last')

# Join
sepsis_ts_df = df_num.join(df_notes, how='outer')

# 2. Resample to Hourly Intervals
# We separate numeric and text columns to apply different aggregation functions during resampling
numeric_cols = [c for c in sepsis_ts_df.columns if c != 'Clinical Note']
text_cols = ['Clinical Note']

# Resample Numeric (Mean)
# Using groupby level 0 (patient_id) and resampling level 1 (timestamp)
# Changed '1H' to '1h' to avoid FutureWarning
resampled_num = sepsis_ts_df[numeric_cols].groupby(level=0).resample('1h', level=1).mean()

# Resample Text (Last entry in the hour)
# Changed '1H' to '1h' to avoid FutureWarning
resampled_note = sepsis_ts_df[text_cols].groupby(level=0).resample('1h', level=1).last()

# Combine and Drop the extra index level created by groupby-resample if necessary
# Pandas typically returns (patient_id, timestamp) index, which is what we want.
sepsis_ts_df = pd.concat([resampled_num, resampled_note], axis=1)

# 3. Forward Fill Missing Values
# Fill forward within each patient group to simulate 'last known value'
sepsis_ts_df = sepsis_ts_df.groupby(level=0).ffill()

# 4. Implement Simplified SOFA Score
# Initialize score
sofa_score = pd.Series(0, index=sepsis_ts_df.index)

# Vectorized Scoring Logic
# Coagulation (Platelets)
sofa_score += (sepsis_ts_df['Platelets'] < 150).astype(int) + (sepsis_ts_df['Platelets'] < 100).astype(int)

# Liver (Bilirubin)
sofa_score += (sepsis_ts_df['Bilirubin'] > 1.2).astype(int) + (sepsis_ts_df['Bilirubin'] > 2.0).astype(int)

# Renal (Creatinine)
sofa_score += (sepsis_ts_df['Creatinine'] > 1.2).astype(int) + (sepsis_ts_df['Creatinine'] > 2.0).astype(int)

# Cardiovascular (SBP)
sofa_score += (sepsis_ts_df['Systolic BP'] < 100).astype(int)

# Assign score
sepsis_ts_df['sofa_score'] = sofa_score

# 5. Detect Suspected Infection
# Fill NaN notes for string search
sepsis_ts_df['Clinical Note'] = sepsis_ts_df['Clinical Note'].fillna('')

keywords = ['antibiotics', 'infection', 'culture', 'sepsis']
pattern = '|'.join(keywords)

# Flag individual rows
sepsis_ts_df['infection_flag'] = sepsis_ts_df['Clinical Note'].str.contains(pattern, case=False, na=False).astype(int)

# Create cumulative suspicion (once suspected, stays suspected)
sepsis_ts_df['infection_suspicion'] = sepsis_ts_df.groupby(level=0)['infection_flag'].cummax()

# 6. Generate Ground Truth Labels
# Sepsis Criteria: Infection Suspicion AND SOFA >= 2
sepsis_ts_df['meets_sepsis_criteria'] = ((sepsis_ts_df['infection_suspicion'] == 1) & (sepsis_ts_df['sofa_score'] >= 2)).astype(int)

# Label 'is_septic': 1 from the first onset onwards
sepsis_ts_df['is_septic'] = sepsis_ts_df.groupby(level=0)['meets_sepsis_criteria'].cummax()

# 7. Print Results
total_patients = sepsis_ts_df.index.get_level_values(0).nunique()
septic_patients = sepsis_ts_df[sepsis_ts_df['is_septic'] == 1].index.get_level_values(0).nunique()

print(f"Total Patients Processed: {total_patients}")
print(f"Identified Septic Patients: {septic_patients}")
print(f"Sepsis Prevalence: {septic_patients/total_patients:.2%}")

print("\nFirst few rows of processed data:")
sepsis_ts_df.head()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import numpy as np
import pandas as pd

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
unique_patients = sepsis_ts_df.index.get_level_values(0).unique()

# First split: Train (70%) and Temp (30%)
train_ids, temp_ids = train_test_split(unique_patients, test_size=0.30, random_state=42)

# Second split: Validation (15% of total) and Test (15% of total) from Temp
# Since Temp is 30% of total, splitting it 50/50 gives 15% of total each.
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42)

print(f"Training set: {len(train_ids)} patients")
print(f"Validation set: {len(val_ids)} patients")
print(f"Testing set: {len(test_ids)} patients")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").to(device)
model.eval()
print("Bio_ClinicalBERT loaded successfully.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --- 1. Generate Embeddings for Clinical Notes ---

# Extract unique notes to minimize computation (high redundancy in synthetic data)
unique_notes = sepsis_ts_df['Clinical Note'].unique()
print(f"Unique notes to encode: {len(unique_notes)}")

def get_embeddings(text_list, batch_size=32):
    all_embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch_texts = text_list[i:i+batch_size]

        # Tokenize
        # Ensure input is a list of strings
        inputs = tokenizer(list(batch_texts), padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)

        # Inference
        with torch.no_grad():
            outputs = model(**inputs)

        # Use CLS token (index 0) as the sentence embedding
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)

# Generate embeddings
if len(unique_notes) > 0:
    note_embeddings = get_embeddings(unique_notes)
    # Create a lookup dictionary
    embedding_map = {note: emb for note, emb in zip(unique_notes, note_embeddings)}

    # Map back to the dataframe
    # We store the embedding vectors in a new object-dtype column
    sepsis_ts_df['note_embedding'] = sepsis_ts_df['Clinical Note'].map(embedding_map)
    print("Embeddings generated and mapped.")
else:
    print("No notes found.")

# --- 2. Standardize Numerical Features ---

feature_cols = ['Bilirubin', 'Creatinine', 'Heart Rate', 'Lactate', 'Platelets', 'Systolic BP', 'Temperature']
scaler = StandardScaler()

# Identify training rows based on patient_id
train_mask = sepsis_ts_df.index.get_level_values('patient_id').isin(train_ids)

# Fit scaler ONLY on training data to prevent leakage
scaler.fit(sepsis_ts_df.loc[train_mask, feature_cols])

# Transform the entire dataset
sepsis_ts_df[feature_cols] = scaler.transform(sepsis_ts_df[feature_cols])

print("Numerical features standardized.")

# --- 3. Final Check ---
print(f"Processed DataFrame Shape: {sepsis_ts_df.shape}")
print("Columns:", sepsis_ts_df.columns.tolist())
sepsis_ts_df.head()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# --- 1. Define Custom Dataset ---
class SepsisDataset(Dataset):
    def __init__(self, df, patient_ids, max_len=50, embedding_dim=768):
        self.df = df
        self.patient_ids = patient_ids
        self.max_len = max_len
        self.embedding_dim = embedding_dim
        self.feature_cols = ['Bilirubin', 'Creatinine', 'Heart Rate', 'Lactate', 'Platelets', 'Systolic BP', 'Temperature']

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        # Access data for specific patient
        patient_data = self.df.loc[pid]

        # --- Numerical Features ---
        x_num = patient_data[self.feature_cols].values.astype(np.float32)

        # Padding/Truncating to fixed length
        seq_len = len(x_num)
        if seq_len < self.max_len:
            # Pad with zeros at the end
            pad_len = self.max_len - seq_len
            x_num = np.pad(x_num, ((0, pad_len), (0, 0)), mode='constant')
        else:
            # Truncate to keep the most recent data (last max_len steps)
            x_num = x_num[-self.max_len:]

        # --- Note Embeddings ---
        # We take the embedding from the last timestamp to represent the most current clinical context
        # Handle cases where 'note_embedding' might be missing or NaN
        x_text = np.zeros(self.embedding_dim, dtype=np.float32)

        if 'note_embedding' in patient_data.columns:
            # Get the series of embeddings
            emb_series = patient_data['note_embedding']

            # Find the last valid embedding (non-NaN)
            # Since we ffilled, looking at the last row is usually sufficient,
            # but checking for validity is safer.
            last_valid_idx = emb_series.last_valid_index()
            if last_valid_idx is not None:
                val = emb_series.loc[last_valid_idx]
                # Ensure it's a valid array
                if isinstance(val, (np.ndarray, list)):
                    x_text = np.array(val, dtype=np.float32)

        # --- Label ---
        # Binary classification: Is the patient septic? (Cumulative label 1)
        y = patient_data['is_septic'].max()

        return (
            torch.tensor(x_num, dtype=torch.float32),
            torch.tensor(x_text, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )

# --- 2. Instantiate Datasets and DataLoaders ---

# Ensure IDs are available (re-using variables from previous steps)
# If variables are missing in this context, we re-derive them from the split logic if needed,
# but assuming continuity from previous cells.

train_dataset = SepsisDataset(sepsis_ts_df, train_ids)
val_dataset = SepsisDataset(sepsis_ts_df, val_ids)
test_dataset = SepsisDataset(sepsis_ts_df, test_ids)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("DataLoaders created.")

# --- 3. Define Hybrid LSTM-BERT Model ---

class HybridModel(nn.Module):
    def __init__(self, num_features, embedding_dim, hidden_dim=64):
        super(HybridModel, self).__init__()

        # LSTM for Time-Series Numerical Data
        self.lstm = nn.LSTM(input_size=num_features, hidden_size=hidden_dim, batch_first=True)

        # Fully Connected Layers for Classification
        # Input = LSTM Hidden State + BERT Embedding
        combined_dim = hidden_dim + embedding_dim

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x_num, x_text):
        # x_num: (batch, seq_len, num_features)
        # x_text: (batch, embedding_dim)

        # LSTM Forward Pass
        # out: (batch, seq_len, hidden_dim)
        # hn: (1, batch, hidden_dim) - Last hidden state
        _, (hn, _) = self.lstm(x_num)

        # Take the last layer's hidden state
        lstm_feature = hn[-1] # (batch, hidden_dim)

        # Concatenate LSTM features with Text Embeddings
        combined = torch.cat((lstm_feature, x_text), dim=1)

        # Classification
        output = self.classifier(combined)
        return output.squeeze() # (batch,)

# --- 4. Initialize Model ---

num_features = len(train_dataset.feature_cols)
embedding_dim = 768

model = HybridModel(num_features=num_features, embedding_dim=embedding_dim).to(device)

print("\nModel Architecture:")
print(model)

In [ ]:
import torch.optim as optim

# --- 1. Define Loss and Optimizer ---
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- 2. Training Loop ---
num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    # Training Phase
    for x_num, x_text, y in train_loader:
        # Move to device
        x_num = x_num.to(device)
        x_text = x_text.to(device)
        y = y.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(x_num, x_text)

        # Calculate loss
        loss = criterion(outputs, y)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation Phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_num, x_text, y in val_loader:
            x_num = x_num.to(device)
            x_text = x_text.to(device)
            y = y.to(device)

            outputs = model(x_num, x_text)
            loss = criterion(outputs, y)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

In [ ]:
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
import matplotlib.pyplot as plt

# 1. Inference on Test Set
model.eval()
y_true = []
y_scores = []

with torch.no_grad():
    for x_num, x_text, y in test_loader:
        x_num = x_num.to(device)
        x_text = x_text.to(device)
        y = y.to(device)

        outputs = model(x_num, x_text)

        # Move to CPU for numpy conversion
        y_true.extend(y.cpu().numpy())
        y_scores.extend(outputs.cpu().numpy())

y_true = np.array(y_true)
y_scores = np.array(y_scores)

# 2. Calculate Metrics
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = auc(recall, precision)
roc_auc = roc_auc_score(y_true, y_scores)

print(f"Test ROC-AUC: {roc_auc:.4f}")
print(f"Test AUPRC: {auprc:.4f}")

# 3. Visualize Precision-Recall Curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.', label=f'Hybrid LSTM-BERT (AUPRC = {auprc:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Define Threshold
threshold = 0.5

# 2. Calculate Lead Time
lead_times = []

# Group test data by patient
# We need the original timestamps to calculate hours, but our dataset is index-based tensors.
# We will reconstruct the timeline logic using the fact that data is hourly resampled.
# Since the test_loader shuffles=False and we built it from test_ids, we can iterate through test_ids.

model.eval()

# We need to access data patient by patient to get temporal sequence
# The DataLoader batches data, but for lead time, we need the full sequence per patient.
# So we will iterate through the test_dataset indices corresponding to patient groups.

# Accessing the dataframe directly is easier for this temporal logic
lead_time_patients = []

with torch.no_grad():
    for pid in test_ids:
        # Get patient data
        patient_data = sepsis_ts_df.loc[pid]

        # Check if patient is actually septic
        if patient_data['is_septic'].max() == 1:
            # Find Ground Truth Sepsis Onset Time
            # is_septic is 0 then turns 1. Find the first index where it is 1.
            sepsis_onset_idx = patient_data['is_septic'].idxmax()

            # We need the row number relative to the start of this patient's data
            # Since idxmax returns the timestamp index, we need to find its integer position
            # or just compare timestamps if we process sequentially.

            # Let's process row by row for this patient to simulate real-time monitoring
            # Prepare inputs

            # We can re-use the dataset logic but for a single patient's full sequence
            # However, the model takes fixed length sequences (max_len=50).
            # To simulate "real-time", we should ideally predict at each hour.
            # For simplicity in this metric calculation, we will use the sequences generated by the Dataset,
            # which represent the state at the END of the stay (or truncated).

            # Limitation: The current SepsisDataset returns ONE sample per patient (the last window).
            # To calculate Lead Time properly, we technically need predictions at EACH hour.
            # Let's approximate:
            # We will grab the full data for the patient, and run the model in a sliding window fashion
            # or just feed the growing sequence if the model supports variable length (LSTM does).

            # Efficient approach: Run model on the full sequence for this patient (1, seq_len, features)
            # The model is trained on fixed windows, but LSTMs can handle variable lengths during inference
            # if we adjust the architecture or input handling.
            # Given the `HybridModel` structure takes (batch, seq_len, num_features), it can take variable seq_len.

            # Prepare Data
            x_num_full = patient_data[feature_cols].values.astype(np.float32)

            # Text embedding: We'll use the LAST embedding for simplicity or ffill logic as per dataset
            # For true dynamic prediction, we'd need embedding at each step.
            # Let's simplify: Use the embedding available at the specific step.
            # Since we pre-calculated embeddings into the DF, we can use them.

            # Let's iterate through the hours of the patient stay
            onset_hour = -1
            predicted_hour = -1

            # Get numeric data as tensor
            x_num_tensor = torch.tensor(x_num_full, dtype=torch.float32).unsqueeze(0).to(device)

            # We need to determine the onset hour index (0 to len-1)
            # 'is_septic' is a series.
            is_septic_series = patient_data['is_septic'].values
            onset_hour = np.argmax(is_septic_series) # First index of 1

            # Now find prediction time
            # We will pass the data up to time 't' to the model.
            # To be efficient, we won't loop every single hour if the sequence is long,
            # but for 5000 patients it might be slow.
            # Optimization: The LSTM returns hidden states for ALL timesteps if we don't take [-1].
            # But our specific `HybridModel` forward takes `hn[-1]`.
            # We can modify the logic or just loop.

            found_prediction = False

            # Iterate up to the onset time (we want to know if we predicted BEFORE onset)
            # We check every hour.
            for t in range(1, len(patient_data)):
                # Data up to time t
                # Truncate to max_len if needed to match training distribution,
                # but LSTM can handle shorter/longer. Let's stick to max_len constraint if t > 50
                start_idx = max(0, t - 50)
                curr_x_num = x_num_full[start_idx:t]

                # Pad if t < 50? The model handles variable length in batch_first=True?
                # Actually, our dataset padded.
                # Let's pad for consistency.
                if len(curr_x_num) < 50:
                    pad_len = 50 - len(curr_x_num)
                    curr_x_num = np.pad(curr_x_num, ((0, pad_len), (0, 0)), mode='constant')

                curr_x_num_tensor = torch.tensor(curr_x_num, dtype=torch.float32).unsqueeze(0).to(device)

                # Get embedding at time t (or most recent)
                # We need to look at the dataframe row t-1 (since slice is :t, exclusive)
                # Handle missing embeddings
                curr_emb = np.zeros(768, dtype=np.float32)
                # Check column
                if 'note_embedding' in patient_data.columns:
                     # We need valid embedding up to this point
                     subset = patient_data['note_embedding'].iloc[0:t]
                     last_valid = subset.last_valid_index()
                     if last_valid is not None:
                         val = patient_data['note_embedding'].loc[last_valid]
                         if isinstance(val, (np.ndarray, list)):
                             curr_emb = np.array(val, dtype=np.float32)

                curr_x_text_tensor = torch.tensor(curr_emb, dtype=torch.float32).unsqueeze(0).to(device)

                # Predict
                prob = model(curr_x_num_tensor, curr_x_text_tensor).item()

                if prob > threshold:
                    predicted_hour = t
                    found_prediction = True
                    break

            # Calculate Lead Time
            if found_prediction and predicted_hour < onset_hour:
                # Lead time in hours
                lead = onset_hour - predicted_hour
                lead_times.append(lead)
                lead_time_patients.append(pid)

# 3. Compute and Print Average
if lead_times:
    avg_lead_time = np.mean(lead_times)
    print(f"Average Lead Time for Correct Predictions: {avg_lead_time:.2f} hours")
    print(f"Number of cases with positive lead time: {len(lead_times)}")
else:
    print("No successful early predictions found.")

# 4. Histogram
plt.figure(figsize=(8, 6))
plt.hist(lead_times, bins=15, color='skyblue', edgecolor='black')
plt.xlabel('Lead Time (Hours)')
plt.ylabel('Count')
plt.title('Distribution of Prediction Lead Times')
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- 1. Adjust Threshold ---
# The model output probabilities are low due to class imbalance.
# We'll use a lower threshold to demonstrate the lead time metric functionality.
# Let's verify the distribution of scores first to pick a sensible one.
print(f"Max prediction score in test set: {np.max(y_scores):.4f}")
print(f"Mean prediction score in test set: {np.mean(y_scores):.4f}")

# Set threshold slightly above mean or at a specific quantile to capture top predictions
# For demonstration, we'll try a lower threshold.
adjusted_threshold = np.percentile(y_scores, 95) # Top 5% of predictions
print(f"Adjusted Threshold (95th percentile): {adjusted_threshold:.4f}")

# --- 2. Recalculate Lead Time ---
lead_times = []
model.eval()

with torch.no_grad():
    for pid in test_ids:
        # Get patient data
        patient_data = sepsis_ts_df.loc[pid]

        # Check if patient is actually septic
        if patient_data['is_septic'].max() == 1:
            # Find Ground Truth Sepsis Onset Time (first index where is_septic == 1)
            sepsis_onset_idx = patient_data['is_septic'].values.argmax()

            # Skip if sepsis is present at admission (index 0)
            if sepsis_onset_idx == 0:
                continue

            # Prepare Data
            x_num_full = patient_data[feature_cols].values.astype(np.float32)

            # Iterate up to the onset time to check for early warning
            found_early_warning = False
            first_alarm_time = -1

            for t in range(1, sepsis_onset_idx):
                # Prepare input window (taking up to 50 previous time steps)
                start_idx = max(0, t - 50)
                curr_x_num = x_num_full[start_idx:t]

                # Pad if sequence is shorter than 50
                if len(curr_x_num) < 50:
                    pad_len = 50 - len(curr_x_num)
                    curr_x_num = np.pad(curr_x_num, ((0, pad_len), (0, 0)), mode='constant')

                curr_x_num_tensor = torch.tensor(curr_x_num, dtype=torch.float32).unsqueeze(0).to(device)

                # Get embedding (simplified: use last valid)
                curr_emb = np.zeros(768, dtype=np.float32)
                if 'note_embedding' in patient_data.columns:
                     subset = patient_data['note_embedding'].iloc[0:t]
                     last_valid = subset.last_valid_index()
                     if last_valid is not None:
                         val = patient_data['note_embedding'].loc[last_valid]
                         if isinstance(val, (np.ndarray, list)):
                             curr_emb = np.array(val, dtype=np.float32)

                curr_x_text_tensor = torch.tensor(curr_emb, dtype=torch.float32).unsqueeze(0).to(device)

                # Predict
                prob = model(curr_x_num_tensor, curr_x_text_tensor).item()

                if prob > adjusted_threshold:
                    first_alarm_time = t
                    found_early_warning = True
                    break

            if found_early_warning:
                lead_time = sepsis_onset_idx - first_alarm_time
                lead_times.append(lead_time)

# --- 3. Results ---
if lead_times:
    avg_lead_time = np.mean(lead_times)
    print(f"\nAnalysis with adjusted threshold ({adjusted_threshold:.4f}):")
    print(f"Average Lead Time: {avg_lead_time:.2f} hours")
    print(f"Number of successful early predictions: {len(lead_times)}")

    # Histogram
    plt.figure(figsize=(8, 6))
    plt.hist(lead_times, bins=10, color='lightgreen', edgecolor='black')
    plt.xlabel('Lead Time (Hours)')
    plt.ylabel('Frequency')
    plt.title(f'Lead Time Distribution (Threshold={adjusted_threshold:.3f})')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print(f"No early predictions found even with threshold {adjusted_threshold:.4f}.")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# 1. Define Thresholds
threshold_05 = 0.5
threshold_95 = np.percentile(y_scores, 95)

print(f"Threshold 0.5: {threshold_05}")
print(f"Threshold 95th Percentile: {threshold_95:.4f}")

# 2. Define Calculation Function
def calculate_lead_metrics(threshold, model, test_ids, df, feature_cols, device='cuda'):
    lead_times = []
    model.eval()

    with torch.no_grad():
        for pid in test_ids:
            # Get patient data
            patient_data = df.loc[pid]

            # Check if patient is actually septic
            if patient_data['is_septic'].max() == 1:
                # Find Ground Truth Sepsis Onset Time
                # Use .values to avoid index alignment issues if index is not unique monotonic
                is_septic_values = patient_data['is_septic'].values
                sepsis_onset_idx = np.argmax(is_septic_values)

                # Skip if sepsis is present at admission
                if sepsis_onset_idx == 0:
                    continue

                # Prepare Data
                x_num_full = patient_data[feature_cols].values.astype(np.float32)

                found_early_warning = False
                first_alarm_time = -1

                # Iterate up to the onset time
                for t in range(1, sepsis_onset_idx):
                    # Prepare input window (max 50 steps)
                    start_idx = max(0, t - 50)
                    curr_x_num = x_num_full[start_idx:t]

                    # Pad if needed
                    if len(curr_x_num) < 50:
                        pad_len = 50 - len(curr_x_num)
                        curr_x_num = np.pad(curr_x_num, ((0, pad_len), (0, 0)), mode='constant')

                    curr_x_num_tensor = torch.tensor(curr_x_num, dtype=torch.float32).unsqueeze(0).to(device)

                    # Get embedding (simplified: use last valid)
                    curr_emb = np.zeros(768, dtype=np.float32)
                    if 'note_embedding' in patient_data.columns:
                         subset = patient_data['note_embedding'].iloc[0:t]
                         last_valid = subset.last_valid_index()
                         if last_valid is not None:
                             val = patient_data['note_embedding'].loc[last_valid]
                             if isinstance(val, (np.ndarray, list)):
                                 curr_emb = np.array(val, dtype=np.float32)

                    curr_x_text_tensor = torch.tensor(curr_emb, dtype=torch.float32).unsqueeze(0).to(device)

                    # Predict
                    prob = model(curr_x_num_tensor, curr_x_text_tensor).item()

                    if prob > threshold:
                        first_alarm_time = t
                        found_early_warning = True
                        break

                if found_early_warning:
                    lead_time = sepsis_onset_idx - first_alarm_time
                    lead_times.append(lead_time)

    return lead_times

# 3. Run for both thresholds
print("\nCalculating metrics for Threshold=0.5...")
lead_times_05 = calculate_lead_metrics(threshold_05, model, test_ids, sepsis_ts_df, feature_cols, device)

print(f"Calculating metrics for Threshold={threshold_95:.4f}...")
lead_times_95 = calculate_lead_metrics(threshold_95, model, test_ids, sepsis_ts_df, feature_cols, device)

# 4. Compare Results
avg_lead_05 = np.mean(lead_times_05) if lead_times_05 else 0
avg_lead_95 = np.mean(lead_times_95) if lead_times_95 else 0

print("\n--- Comparison ---")
print(f"Threshold 0.5:   Avg Lead Time = {avg_lead_05:.2f} h, Successful Detections = {len(lead_times_05)}")
print(f"Threshold 95th%: Avg Lead Time = {avg_lead_95:.2f} h, Successful Detections = {len(lead_times_95)}")

# 5. Plot Histogram for Best Threshold
best_lead_times = lead_times_95 if len(lead_times_95) > len(lead_times_05) else lead_times_05
best_thresh_name = "95th Percentile" if len(lead_times_95) > len(lead_times_05) else "0.5"
best_thresh_val = threshold_95 if len(lead_times_95) > len(lead_times_05) else threshold_05

if best_lead_times:
    plt.figure(figsize=(8, 6))
    plt.hist(best_lead_times, bins=15, color='orange', edgecolor='black')
    plt.xlabel('Lead Time (Hours)')
    plt.ylabel('Count')
    plt.title(f'Lead Time Distribution (Best Threshold: {best_thresh_name} = {best_thresh_val:.3f})')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No successful detections to plot.")

# Task
Implement a robust training loop for the `HybridModel` using `EarlyStopping` and a `ReduceLROnPlateau` learning rate scheduler.

Specifically, perform the following:
1.  **Re-initialize** the `HybridModel`, optimizer (Adam, lr=1e-3), and loss function (BCELoss) to ensure a fresh start.
2.  **Define an `EarlyStopping` class** (or logic) that monitors validation loss, saves the model checkpoint (`best_model.pth`) when loss improves, and stops training if loss doesn't improve for `patience=5` epochs.
3.  **Initialize a Scheduler**: Use `torch.optim.lr_scheduler.ReduceLROnPlateau` to reduce the learning rate by a factor of 0.5 if validation loss stagnates for 2 epochs.
4.  **Execute Training**: Run the training loop for up to 30 epochs. Print the Train Loss, Validation Loss, and current Learning Rate for each epoch.
5.  **Load Best Weights**: After training finishes (or stops early), load the saved `best_model.pth` weights back into the model.

```python
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

# --- 1. Early Stopping Utility ---
class EarlyStopping:
    def __init__(self, patience=5, delta=0, path='best_model.pth'):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
        elif val_loss > self.best_loss + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        print(f'Validation loss decreased ({self.best_loss:.6f} --> {val_loss:.6f}).  Saving model ...')
        self.best_loss = val_loss

# --- 2. Re-initialize Model and Setup ---
# Re-init model to start fresh
model = HybridModel(num_features=num_features, embedding_dim=embedding_dim).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Scheduler: Reduce LR when validation loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

# Early Stopping
early_stopping = EarlyStopping(patience=5, path='best_model.pth')

# --- 3. Advanced Training Loop ---
num_epochs = 30
train_losses = []
val_losses = []

print("Starting Advanced Training...")

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    for x_num, x_text, y in train_loader:
        x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x_num, x_text)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation Phase ---
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for x_num, x_text, y in val_loader:
            x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)
            outputs = model(x_num, x_text)
            loss = criterion(outputs, y)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Get current LR
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.6f}")

    # --- Scheduler Step ---
    scheduler.step(avg_val_loss)

    # --- Early Stopping Check ---
    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

# --- 4. Load Best Model ---
print("\nLoading best model weights...")
model.load_state_dict(torch.load('best_model.pth'))
print("Model loaded successfully.")
```

## Advanced Model Training

### Subtask:
Train the HybridModel using EarlyStopping and a learning rate scheduler to improve performance and prevent overfitting.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import copy

# --- 1. Define EarlyStopping Class ---
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, path='best_model.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        # Save model state
        torch.save(model.state_dict(), self.path)
        self.best_model_state = copy.deepcopy(model.state_dict())

# --- 2. Re-initialize Model, Criterion, Optimizer ---
# We re-initialize to start training from scratch
model = HybridModel(num_features=num_features, embedding_dim=embedding_dim).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- 3. Initialize Scheduler ---
# Removed verbose=True to avoid TypeError in some PyTorch versions
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
early_stopping = EarlyStopping(patience=5, path='best_model.pth')

# --- 4. Training Loop with Early Stopping & Scheduler ---
num_epochs = 30
train_losses = []
val_losses = []

print("Starting training with Early Stopping and Scheduler...")

for epoch in range(num_epochs):
    # -- Training Phase --
    model.train()
    running_train_loss = 0.0
    for x_num, x_text, y in train_loader:
        x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x_num, x_text)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # -- Validation Phase --
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for x_num, x_text, y in val_loader:
            x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)
            outputs = model(x_num, x_text)
            loss = criterion(outputs, y)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # -- Print Status --
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.6f}")

    # -- Step Scheduler --
    scheduler.step(avg_val_loss)

    # -- Early Stopping Check --
    early_stopping(avg_val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

# --- 5. Load Best Model ---
# Ensure we load the state dict that was saved
if early_stopping.best_loss is not None:
    model.load_state_dict(torch.load('best_model.pth'))
    print("Best model loaded.")
else:
    print("No model saved (training might have failed or stopped too early).")

## Threshold Sensitivity Analysis

### Subtask:
Analyze the model's performance on the test set across a range of thresholds (0.05 to 0.95) to evaluate the trade-offs between Precision, Recall, and F1-Score.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np
import torch

# 1. Set model to evaluation mode
model.eval()

# 2. Generate predictions for the test set
y_true_test = []
y_scores_test = []

with torch.no_grad():
    for x_num, x_text, y in test_loader:
        x_num = x_num.to(device)
        x_text = x_text.to(device)
        y = y.to(device)

        outputs = model(x_num, x_text)

        y_true_test.extend(y.cpu().numpy())
        y_scores_test.extend(outputs.cpu().numpy())

y_true_test = np.array(y_true_test)
y_scores_test = np.array(y_scores_test)

# 3. Define thresholds
thresholds = np.arange(0.05, 1.00, 0.05)
results = []

# 4. Iterate through thresholds
for thresh in thresholds:
    # a. Convert to binary predictions
    y_pred = (y_scores_test > thresh).astype(int)

    # b. Calculate metrics
    precision = precision_score(y_true_test, y_pred, zero_division=0)
    recall = recall_score(y_true_test, y_pred, zero_division=0)
    f1 = f1_score(y_true_test, y_pred, zero_division=0)

    # c. Calculate TP and FP
    cm = confusion_matrix(y_true_test, y_pred)
    # Confusion matrix structure: [[TN, FP], [FN, TP]]
    # Handle edge cases where confusion matrix might not be 2x2 (e.g. all 0s or all 1s)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        # Simplified fallback if only one class is predicted
        tn = fp = fn = tp = 0
        if len(np.unique(y_true_test)) == 1:
             # This part is just a safe guard, likely won't happen with reasonable data
             pass

    # d. Store results
    results.append({
        'Threshold': round(thresh, 2),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1 Score': round(f1, 4),
        'True Positives': tp,
        'False Positives': fp
    })

# 5. Create DataFrame and display
threshold_df = pd.DataFrame(results)
print("Threshold Sensitivity Analysis:")
display(threshold_df)

In [ ]:
import matplotlib.pyplot as plt

# Plot Precision, Recall, and F1 Score vs Threshold
plt.figure(figsize=(10, 6))
plt.plot(threshold_df['Threshold'], threshold_df['Precision'], marker='o', label='Precision')
plt.plot(threshold_df['Threshold'], threshold_df['Recall'], marker='s', label='Recall')
plt.plot(threshold_df['Threshold'], threshold_df['F1 Score'], marker='^', label='F1 Score')

plt.title('Metric Sensitivity to Classification Threshold')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend()
plt.grid(True, alpha=0.5)
plt.xticks(np.arange(0, 1.05, 0.05))
plt.show()

## Implement Weighted Random Sampler

### Subtask:
Calculate sample weights to balance the class distribution and create a new training DataLoader using WeightedRandomSampler.


In [ ]:
from torch.utils.data import WeightedRandomSampler, DataLoader
import torch
import numpy as np

# 1. Extract Targets for Training Set
# We need to know the label (0 or 1) for each patient in the training set to assign weights.
train_targets = []
for pid in train_ids:
    # Get max label (1 if ever septic, 0 otherwise)
    label = sepsis_ts_df.loc[pid]['is_septic'].max()
    train_targets.append(label)

train_targets = np.array(train_targets)

# 2. Calculate Class Counts
class_counts = np.bincount(train_targets)
print(f"Class Counts (0, 1): {class_counts}")

# 3. Compute Class Weights (Inverse of Frequency)
# weight = 1.0 / count
class_weights = 1.0 / class_counts
print(f"Class Weights: {class_weights}")

# 4. Assign Weight to Each Sample
sample_weights = [class_weights[label] for label in train_targets]
sample_weights = torch.DoubleTensor(sample_weights)

# 5. Create WeightedRandomSampler
# num_samples = len(sample_weights) ensures we draw the same number of samples per epoch
# replacement=True is required for oversampling the minority class
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# 6. Create Balanced DataLoader
# Note: shuffle=False is mandatory when using a sampler
train_loader_balanced = DataLoader(train_dataset, batch_size=32, sampler=sampler)

# 7. Verify Balancing
# Fetch one batch to check the distribution
data_iter = iter(train_loader_balanced)
x_num, x_text, y = next(data_iter)

print(f"\nBatch Size: {len(y)}")
print(f"Number of Septic Patients in Batch: {y.sum().item()}")
print(f"Proportion of Septic Patients: {y.mean().item():.2f}")

In [ ]:
import copy

# --- 1. Re-initialize Model Components ---
model = HybridModel(num_features=num_features, embedding_dim=embedding_dim).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Scheduler & Early Stopping
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
early_stopping = EarlyStopping(patience=5, path='best_model_balanced.pth')

# --- 2. Retraining Loop with Balanced Loader ---
num_epochs = 30
train_losses = []
val_losses = []

print("Starting training with Balanced Data...")

for epoch in range(num_epochs):
    # -- Training Phase (Using Balanced Loader) --
    model.train()
    running_train_loss = 0.0
    for x_num, x_text, y in train_loader_balanced:
        x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x_num, x_text)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader_balanced)
    train_losses.append(avg_train_loss)

    # -- Validation Phase --
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for x_num, x_text, y in val_loader:
            x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)
            outputs = model(x_num, x_text)
            loss = criterion(outputs, y)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # -- Print Status --
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {current_lr:.6f}")

    # -- Scheduler & Early Stopping --
    scheduler.step(avg_val_loss)
    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

# --- 3. Load Best Balanced Model ---
model.load_state_dict(torch.load('best_model_balanced.pth'))
print("Best balanced model loaded.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np
import torch

# --- 1. Generate Predictions on Test Set ---
model.eval()
y_true_test = []
y_scores_test = []

with torch.no_grad():
    for x_num, x_text, y in test_loader:
        x_num = x_num.to(device)
        x_text = x_text.to(device)
        y = y.to(device)

        outputs = model(x_num, x_text)

        y_true_test.extend(y.cpu().numpy())
        y_scores_test.extend(outputs.cpu().numpy())

y_true_test = np.array(y_true_test)
y_scores_test = np.array(y_scores_test)

# --- 2. Plot Histogram of Predicted Probabilities ---
plt.figure(figsize=(10, 6))
plt.hist(y_scores_test[y_true_test == 0], bins=50, alpha=0.5, label='Non-Septic', color='blue', density=True)
plt.hist(y_scores_test[y_true_test == 1], bins=50, alpha=0.5, label='Septic', color='red', density=True)
plt.xlabel('Predicted Probability')
plt.ylabel('Density')
plt.title('Distribution of Predicted Probabilities (Balanced Training)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- 3. Re-run Threshold Sensitivity Analysis ---
thresholds = np.arange(0.05, 1.00, 0.05)
results = []

for thresh in thresholds:
    y_pred = (y_scores_test > thresh).astype(int)

    precision = precision_score(y_true_test, y_pred, zero_division=0)
    recall = recall_score(y_true_test, y_pred, zero_division=0)
    f1 = f1_score(y_true_test, y_pred, zero_division=0)

    cm = confusion_matrix(y_true_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0 # Simplified fallback

    results.append({
        'Threshold': round(thresh, 2),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1 Score': round(f1, 4),
        'True Positives': tp,
        'False Positives': fp
    })

balanced_threshold_df = pd.DataFrame(results)
print("Threshold Sensitivity Analysis (Balanced Model):")
display(balanced_threshold_df)